# 11 · Full reproducibility audit

Audit the complete experiment chain and optionally reproduce it without silently hiding failures.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Reproduction is a chain

Configuration generates versioned data; validation checks the data contract; experiments record
metadata and artifacts; tests verify behavior; reports summarize evidence and preserve limitations.

In [ ]:
manifest_path = Path("data/generated/portfolio/manifest.json")
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(manifest["dataset_version"], manifest["rows"])
else:
    print("Portfolio data is generated locally by the reproduction command.")

## Verify committed metric metadata

In [ ]:
audit = []
for path in sorted(Path("artifacts/metrics").glob("*.json")):
    payload = json.loads(path.read_text())
    run = payload.get("_run", {})
    audit.append({"lab": path.stem, "profile": run.get("profile"),
                  "seed": run.get("seed"), "dataset": run.get("dataset_version"),
                  "elapsed": run.get("elapsed_seconds")})
display(pd.DataFrame(audit))

## Artifact integrity

In [ ]:
import hashlib
artifact_rows = []
for path in sorted(Path("artifacts/figures").glob("*portfolio*")):
    artifact_rows.append({"file": path.name, "bytes": path.stat().st_size,
                          "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
display(pd.DataFrame(artifact_rows))

## Optional full run

The flag defaults to false so merely opening the notebook cannot start a long job. Set it only after
checking disk, CPU budget, and the profile. Failures are raised, not converted into green results.

In [ ]:
RUN_FULL_REPRODUCTION = False
if RUN_FULL_REPRODUCTION:
    subprocess.run([sys.executable, "scripts/reproduce_all.py", "--profile", "portfolio"],
                   check=True)
else:
    print("Skipped by explicit safety switch; committed portfolio evidence remains inspectable.")

## Reproducibility report

In [ ]:
subprocess.run([sys.executable, "scripts/build_reproducibility_report.py"], check=True)
print(Path("reports/reproducibility-report.md").read_text())

## Expected variation

Wall time, resident memory, TensorFlow kernels, parallel tree training, and accelerator behavior can
vary. Dataset checksums and configuration should remain stable; metrics should remain within justified
tolerances. Investigate a deviation before accepting or suppressing it.

In [ ]:
print(Path("reports/limitations.md").read_text())

## Exercises

Reproduce on a second machine; compare environment captures; create an acceptance range for ROC-AUC;
change one seed and quantify variance; explain why committing only a model file is not reproducibility.